# Implement Sliding Window Attention — Solution

**Difficulty**: 🟡 Medium

**Companies**: Mistral, Anthropic, Google, DeepMind

---

### Problem Statement

Standard self-attention has **O(n²)** memory and compute complexity, which becomes prohibitive for long sequences. **Sliding window attention** restricts each token to a local window of keys:

- **Bidirectional** (Longformer-style, encoders): token `i` sees the `w` tokens on each side, `i-w … i+w`.
- **Causal** (Mistral-style, decoders): token `i` sees only the window that ends at itself, `i-w … i`.

Either way each query touches O(w) keys, so attention costs **O(n·w)** instead of O(n²) — *if* the implementation skips the masked entries (e.g. FlashAttention's `window_size` argument, or the `unfold` bonus below). Masking a full n×n score matrix, as in the first part, gives the right numbers but not the savings.

Your task is to implement sliding window attention by:
1. Creating a **band-diagonal attention mask** where token `i` can attend to tokens `[i-w, i+w]` — or only `[i-w, i]` when `causal=True`.
2. Applying the mask to **scaled dot-product attention**.
3. Comparing the output with full attention to verify correctness.

---

### Requirements

1. **`create_sliding_window_mask(seq_len, window_size, causal=False)`** — Returns a boolean mask of shape `(seq_len, seq_len)` (rows = queries `i`, columns = keys `j`) where `mask[i][j] = True` iff `|i - j| <= window_size`, and additionally `j <= i` when `causal=True`.
2. **`full_attention(q, k, v)`** — Plain scaled dot-product attention, `softmax(q kᵀ / √d) v`, as the reference.
3. **`sliding_window_attention(q, k, v, window_size, causal=False)`** — Applies scaled dot-product attention with the sliding window mask. Masked positions should get `-inf` before softmax.
4. **Memory comparison** — Show that sliding window uses O(n·w) non-zero entries vs O(n²) for full attention.

---

### Constraints

- ✅ Must handle batched inputs: `q, k, v` have shape `(batch_size, seq_len, d_model)`.
- ✅ When `window_size >= seq_len`, output must match full (unmasked) attention exactly — or ordinary causal attention when `causal=True`.
- ✅ Attention weights for each query must sum to 1 (valid probability distribution).
- ❌ Do **not** use `torch.nn.MultiheadAttention` or any built-in attention layers (the tests use `F.scaled_dot_product_attention` as a reference).

> **Mistral's convention.** Mistral 7B uses *causal* sliding-window attention with W = 4096. The paper describes token `i` attending to positions between `i − W` and `i`; Hugging Face's implementation lets each query see the W most recent positions *including itself* (`i − W < j ≤ i`), which is `window_size = W − 1` in this notebook's convention. Codebases disagree on this off-by-one, so state yours explicitly in an interview. Because each layer looks back W tokens, k stacked layers give a receptive field of roughly k·W, and at inference a rolling KV cache of W entries suffices.

---

<details>
  <summary>💡 Hint</summary>

  - Create the mask from pairwise offsets: `pos = torch.arange(seq_len)`, `offset = pos.unsqueeze(1) - pos.unsqueeze(0)` (query minus key), then `offset.abs() <= window_size`. For `causal=True`, also require `offset >= 0`.
  - Apply the mask with `scores.masked_fill(~mask, float('-inf'))` before softmax (the `(seq_len, seq_len)` mask broadcasts over the batch). The diagonal is always kept, so no row is fully masked.
  - Scale factor for attention: `1 / sqrt(d_model)`.
  - Non-zero entries in a band matrix of width 2w+1: approximately `n * (2w + 1)` (clamped at edges).

</details>

---

In [ ]:
import math
import torch
import torch.nn.functional as F

In [ ]:
# Setup
torch.manual_seed(42)

batch_size = 2
seq_len = 16
d_model = 32
window_size = 3  # attend to 3 tokens on each side

q = torch.randn(batch_size, seq_len, d_model)
k = torch.randn(batch_size, seq_len, d_model)
v = torch.randn(batch_size, seq_len, d_model)

print(f"Q/K/V shape: {q.shape}")
print(f"Window size: {window_size} (each token attends to {2 * window_size + 1} positions)")

In [ ]:
def create_sliding_window_mask(seq_len: int, window_size: int, causal: bool = False) -> torch.Tensor:
    """
    Create a sliding window attention mask.

    Args:
        seq_len:     sequence length
        window_size: number of tokens to attend to on each side (only before, if causal)
        causal:      if True, query i sees keys i - window_size .. i (Mistral-style)

    Returns:
        Boolean tensor of shape (seq_len, seq_len) where True = allowed to attend.
        Rows are queries i, columns are keys j.
    """
    assert window_size >= 0, "window_size must be non-negative"
    positions = torch.arange(seq_len)
    offset = positions.unsqueeze(1) - positions.unsqueeze(0)  # offset[i, j] = i - j
    mask = offset.abs() <= window_size
    if causal:
        mask = mask & (offset >= 0)  # keys at or before the query only
    return mask


def full_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Standard scaled dot-product attention (no mask).

    Args:
        q, k, v: (batch_size, seq_len, d_model)

    Returns:
        Output tensor of shape (batch_size, seq_len, d_model)
    """
    d = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)
    weights = F.softmax(scores, dim=-1)
    return weights @ v


def sliding_window_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                             window_size: int, causal: bool = False) -> torch.Tensor:
    """
    Scaled dot-product attention with sliding window mask.

    Args:
        q, k, v:     (batch_size, seq_len, d_model)
        window_size: number of tokens to attend to on each side (only before, if causal)
        causal:      if True, query i sees keys i - window_size .. i

    Returns:
        Output tensor of shape (batch_size, seq_len, d_model)
    """
    d = q.size(-1)
    seq_len = q.size(1)

    # Step 1: Compute raw attention scores
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)  # (batch, seq_len, seq_len)

    # Step 2: Create the sliding window mask (on the same device as the scores)
    mask = create_sliding_window_mask(seq_len, window_size, causal).to(scores.device)

    # Step 3: Apply mask - set masked (False) positions to -inf BEFORE the softmax.
    # The diagonal is always allowed, so no row is fully masked (no NaNs).
    scores = scores.masked_fill(~mask, float('-inf'))

    # Step 4: Softmax and weighted sum
    weights = F.softmax(scores, dim=-1)
    return weights @ v


# Quick test
mask = create_sliding_window_mask(seq_len, window_size)
print(f"Mask shape: {mask.shape}")
print(f"Mask (first 8x8):\n{mask[:8, :8].int()}")
print(f"Causal mask (first 8x8):\n{create_sliding_window_mask(seq_len, window_size, causal=True)[:8, :8].int()}")

out = sliding_window_attention(q, k, v, window_size)
print(f"\nOutput shape: {out.shape}")

In [ ]:
# Validation

# Validation 1: Mask semantics on a small example (rows = queries i, columns = keys j)
print("=== Mask Semantics ===")
expected = torch.tensor([[1, 1, 0, 0, 0],
                         [1, 1, 1, 0, 0],
                         [0, 1, 1, 1, 0],
                         [0, 0, 1, 1, 1],
                         [0, 0, 0, 1, 1]], dtype=torch.bool)
expected_causal = torch.tensor([[1, 0, 0, 0, 0],
                                [1, 1, 0, 0, 0],
                                [0, 1, 1, 0, 0],
                                [0, 0, 1, 1, 0],
                                [0, 0, 0, 1, 1]], dtype=torch.bool)
assert torch.equal(create_sliding_window_mask(5, 1), expected), "Bidirectional mask is wrong!"
assert torch.equal(create_sliding_window_mask(5, 1, causal=True), expected_causal), "Causal mask is wrong!"
print("PASSED: window_size=1 masks match (bidirectional and causal).\n")

# Validation 2: When window_size >= seq_len, should match full (or plain causal) attention
print("=== Full Window = Full Attention ===")
out_full = full_attention(q, k, v)
out_big_window = sliding_window_attention(q, k, v, window_size=seq_len)
assert torch.allclose(out_full, out_big_window, atol=1e-6), "Large window should match full attention!"
out_causal = F.scaled_dot_product_attention(q, k, v, is_causal=True)
out_big_causal = sliding_window_attention(q, k, v, window_size=seq_len, causal=True)
assert torch.allclose(out_causal, out_big_causal, atol=1e-5), "Large causal window should match causal attention!"
print("PASSED: window_size=seq_len matches full attention (and causal attention when causal=True).\n")

# Validation 3: Agree with PyTorch's fused attention given the same boolean mask
print("=== Matches F.scaled_dot_product_attention ===")
assert torch.allclose(full_attention(q, k, v), F.scaled_dot_product_attention(q, k, v), atol=1e-5), \
    "full_attention should match F.scaled_dot_product_attention!"
for causal in (False, True):
    for w in (0, 1, 3, 7):
        ours = sliding_window_attention(q, k, v, w, causal=causal)
        ref = F.scaled_dot_product_attention(
            q, k, v, attn_mask=create_sliding_window_mask(seq_len, w, causal=causal))
        assert torch.allclose(ours, ref, atol=1e-5), f"Mismatch for window_size={w}, causal={causal}!"
print("PASSED: matches the fused kernel for several window sizes, bidirectional and causal.\n")

# Validation 4: Attention weights — with V = identity, the output IS the weight matrix
print("=== Attention Weights ===")
eye = torch.eye(seq_len).expand(batch_size, seq_len, seq_len)
weights = sliding_window_attention(q, k, eye, window_size)  # (batch, seq_len, seq_len)
assert torch.allclose(weights.sum(dim=-1), torch.ones(batch_size, seq_len), atol=1e-6), "Weights must sum to 1!"
outside = ~create_sliding_window_mask(seq_len, window_size)
assert (weights[:, outside] == 0).all(), "Positions outside the window must get zero weight!"
print("PASSED: every row sums to 1 and positions outside the window get zero weight.\n")

# Validation 5: Memory comparison
print("=== Memory Comparison ===")
for n in [128, 512, 2048, 8192]:
    w = 64
    full_entries = n * n
    m = create_sliding_window_mask(n, w)
    window_entries = m.sum().item()
    ratio = window_entries / full_entries * 100
    print(f"  seq_len={n:5d}, window={w}: {window_entries:>10,.0f} / {full_entries:>12,.0f} entries ({ratio:.1f}%)")
print("PASSED: Sliding window uses far fewer entries for long sequences.\n")

print("All tests passed!")

---

# Bonus: Efficient Sliding Window Attention with `unfold`

The mask-based implementation above still computes the full n×n score matrix. This version uses `torch.Tensor.unfold` to extract sliding windows and **never builds the n×n matrix** — O(n·w·d) compute and O(n·w) score memory. The causal variant only needs the `w` keys before each query, so it pads on the left only.

### How `unfold` works

`unfold(dim, size, step)` slides a window along a dimension and exposes the window contents as a new last dimension. It returns a **view** (no data copy — overlapping windows share memory via strides).

```python
x = torch.arange(10)      # (10,)
x.unfold(0, 4, 1)         # (7, 4) — 7 overlapping windows of size 4
```

---

In [ ]:
def sliding_window_attention_unfold(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                                    window_size: int, causal: bool = False) -> torch.Tensor:
    """
    Efficient sliding window attention using unfold — O(n·w) scores, no n×n matrix.

    Args:
        q, k, v:     (batch_size, seq_len, d_model)
        window_size: number of tokens to attend to on each side (only before, if causal)
        causal:      if True, query i sees keys i - window_size .. i

    Returns:
        Output tensor of shape (batch_size, seq_len, d_model)
    """
    batch, seq_len, d = q.shape
    left = window_size
    right = 0 if causal else window_size
    win = left + right + 1  # keys per query: 2w+1, or w+1 if causal

    # Step 1: Pad k and v along the sequence dimension (left by w, right by w or 0)
    # F.pad args go from last dim to first: (d_left, d_right, seq_left, seq_right)
    k_pad = F.pad(k, (0, 0, left, right))   # (batch, seq_len + left + right, d)
    v_pad = F.pad(v, (0, 0, left, right))

    # Step 2: Extract sliding windows with unfold (a view: windows share memory)
    # unfold(1, win, 1) → (batch, seq_len, d, win); window i covers keys i-left .. i+right
    k_windows = k_pad.unfold(1, win, 1).permute(0, 1, 3, 2)  # (batch, seq_len, win, d)
    v_windows = v_pad.unfold(1, win, 1).permute(0, 1, 3, 2)  # (batch, seq_len, win, d)

    # Step 3: Compute attention scores
    # q: (batch, seq_len, 1, d) @ k_windows^T: (batch, seq_len, d, win) → (batch, seq_len, 1, win)
    scores = (q.unsqueeze(2) @ k_windows.transpose(-2, -1)) / math.sqrt(d)

    # Step 4: Mask out padded positions (they are zero vectors, not real keys)
    # Pad a ones-vector the same way and unfold it: 1 = real key, 0 = padding
    valid = F.pad(q.new_ones(1, seq_len), (left, right))     # (1, seq_len + left + right)
    valid_windows = valid.unfold(1, win, 1).unsqueeze(2)      # (1, seq_len, 1, win)
    scores = scores.masked_fill(valid_windows == 0, float('-inf'))

    # Step 5: Softmax over window dimension and weighted sum
    weights = F.softmax(scores, dim=-1)                  # (batch, seq_len, 1, win)
    out = (weights @ v_windows).squeeze(2)               # (batch, seq_len, d)
    return out


# Quick shape check
out_unfold = sliding_window_attention_unfold(q, k, v, window_size)
print(f"Output shape: {out_unfold.shape}")  # (2, 16, 32)

In [ ]:
# Bonus Validation

# Test 1: Must match mask-based implementation
print("=== Unfold vs Mask-Based ===")
out_mask = sliding_window_attention(q, k, v, window_size)
out_unfold = sliding_window_attention_unfold(q, k, v, window_size)
assert torch.allclose(out_mask, out_unfold, atol=1e-5), \
    f"Outputs differ! Max diff: {(out_mask - out_unfold).abs().max():.2e}"
print("PASSED: unfold version matches mask-based version.\n")

# Test 2: Full window matches full attention
print("=== Unfold Full Window = Full Attention ===")
out_full = full_attention(q, k, v)
out_unfold_full = sliding_window_attention_unfold(q, k, v, window_size=seq_len)
assert torch.allclose(out_full, out_unfold_full, atol=1e-5), "Full window should match full attention!"
print("PASSED: window_size=seq_len matches full attention.\n")

# Test 3: Causal variant matches the causal mask-based version
print("=== Unfold Causal vs Mask-Based Causal ===")
for w in (0, 1, window_size, seq_len):
    out_mask_c = sliding_window_attention(q, k, v, w, causal=True)
    out_unfold_c = sliding_window_attention_unfold(q, k, v, w, causal=True)
    assert torch.allclose(out_mask_c, out_unfold_c, atol=1e-5), \
        f"Causal outputs differ for window_size={w}! Max diff: {(out_mask_c - out_unfold_c).abs().max():.2e}"
print("PASSED: causal unfold version matches the causal mask-based version.\n")

# Test 4: Memory comparison
print("=== Score Entries Computed ===")
for n in [128, 512, 2048, 8192]:
    w = 64
    full_scores = n * n
    unfold_scores = n * (2 * w + 1)
    ratio = unfold_scores / full_scores * 100
    print(f"  seq_len={n:5d}, window={w}: unfold={unfold_scores:>10,} vs full={full_scores:>12,} ({ratio:.1f}%)")
print("PASSED: unfold computes far fewer score entries.\n")

print("All bonus tests passed!")